# How LLMs can learn to sort numbers?

## Generating Dataset

Dataset specifications:
- Integer range: 0–99.
- Sequence length: 2–10 integers.
- No duplicate integers within a sequence.
- Sorting order: ascending.
- Training examples: 20,000.
- Random seed: 42 for reproducibility.

In [1]:
import random
from pathlib import Path


def generate_example(
    rng: random.Random,
    min_length: int,
    max_length: int,
    min_value: int,
    max_value: int,
) -> str:
    length = rng.randint(min_length, max_length)

    numbers = rng.sample(
        range(min_value, max_value + 1),
        length,
    )

    sorted_numbers = sorted(numbers)

    input_sequence = " ".join(map(str, numbers))
    output_sequence = " ".join(map(str, sorted_numbers))

    return f"{input_sequence} | {output_sequence}"


def generate_dataset(
    output_path: Path,
    num_examples: int,
    min_length: int,
    max_length: int,
    min_value: int,
    max_value: int,
    seed: int,
) -> None:
    if num_examples <= 0:
        raise ValueError("num_examples must be greater than zero.")

    if min_length < 2 or max_length < min_length:
        raise ValueError("Invalid sequence length range.")

    if min_value > max_value:
        raise ValueError("min_value cannot exceed max_value.")

    if max_length > max_value - min_value + 1:
        raise ValueError("Not enough distinct integers for max_length.")

    rng = random.Random(seed)

    examples = [
        generate_example(
            rng,
            min_length,
            max_length,
            min_value,
            max_value,
        )
        for _ in range(num_examples)
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(examples) + "\n", encoding="utf-8")

    print(f"Generated {len(examples)} examples.")
    print(f"Dataset saved to: {output_path}")

In [2]:
SEED = 99

generate_dataset(
    output_path=Path(f"data/sorting_{SEED}.txt"),
    num_examples=2_000_000,
    min_length=2,
    max_length=10,
    min_value=0,
    max_value=99,
    seed=SEED,
)

Generated 2000000 examples.
Dataset saved to: data/sorting_99.txt


In [3]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from emergence.data.tokenizer import SortingTokenizer
from emergence.data.datasets import SortingDataset
from emergence.models.transformer.config import TransformerConfig
from emergence.models.transformer.model import TransformerModel
from emergence.training.train import train_model

PROJECT_ROOT = Path.cwd()

DATASET_PATH = PROJECT_ROOT / "data" / f"sorting_{SEED}.txt"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET_PATH)
print("Checkpoints:", CHECKPOINT_DIR)
print("Outputs:", OUTPUT_DIR)

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"

text = DATASET_PATH.read_text(encoding="utf-8")

tokenizer = SortingTokenizer()
encoded_examples = tokenizer.encode(text)

print("Dataset characters:", len(text))
print("Vocabulary size:", tokenizer.vocab_size)
print("Number of examples:", len(encoded_examples))
print("First encoded example:", encoded_examples[0])

CONTEXT_LENGTH = 32

dataset = SortingDataset(
    encoded_examples=encoded_examples,
    context_length=CONTEXT_LENGTH,
    pad_token_id=tokenizer.vocab["<PAD>"],
)
train_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
)

print("Training samples:", len(dataset))
x, y = dataset[0]
print("Input shape:", x.shape)
print("Target shape:", y.shape)
print("Input:", x)
print("Target:", y)
print("Padding ID:", tokenizer.vocab["<PAD>"])

Dataset: /Users/kanjurer/Development/personal/emergence/experiments/llm_sorting/data/sorting_99.txt
Checkpoints: /Users/kanjurer/Development/personal/emergence/experiments/llm_sorting/checkpoints
Outputs: /Users/kanjurer/Development/personal/emergence/experiments/llm_sorting/outputs
Dataset characters: 73578908
Vocabulary size: 103
Number of examples: 2000000
First encoded example: [48, 25, 76, 22, 29, 31, 17, 97, 100, 17, 22, 25, 29, 31, 48, 76, 97, 101]
Training samples: 2000000
Input shape: torch.Size([32])
Target shape: torch.Size([32])
Input: tensor([ 48,  25,  76,  22,  29,  31,  17,  97, 100,  17,  22,  25,  29,  31,
         48,  76,  97, 102, 102, 102, 102, 102, 102, 102, 102, 102, 102, 102,
        102, 102, 102, 102])
Target: tensor([ 25,  76,  22,  29,  31,  17,  97, 100,  17,  22,  25,  29,  31,  48,
         76,  97, 101, 102, 102, 102, 102, 102, 102, 102, 102, 102, 102, 102,
        102, 102, 102, 102])
Padding ID: 102


## Training

In [4]:
config = TransformerConfig(
    vocab_size=tokenizer.vocab_size,
    context_length=CONTEXT_LENGTH,
    embedding_dim=128,
    num_layers=4,
    num_heads=4,
    dropout=0.1,
)

model = TransformerModel(config)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model.to(device)

print("Device:", device)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

Device: mps
Model parameters: 823808


In [ ]:
run = train_model(
    model=model,
    train_loader=train_loader,
    epochs=15,
    learning_rate=3e-4,
    checkpoint_dir=CHECKPOINT_DIR,
    pad_token_id=tokenizer.vocab["<PAD>"],
)

print("Training complete!")
print("Best loss:", run.best_loss)
print("Completed epochs:", run.completed_epochs)

## Plotting and Results

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(run.history) + 1), run.history)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Sorting Experiment Training Loss")
plt.grid(True)
plt.show()

In [ ]:
import json

results = {
    "epochs": run.completed_epochs,
    "best_loss": run.best_loss,
    "history": run.history,
    "config": {
        "context_length": CONTEXT_LENGTH,
        "embedding_dim": 128,
        "num_layers": 4,
        "num_heads": 4,
        "dropout": 0.1,
        "learning_rate": 3e-4,
        "batch_size": 32,
    },
}

(OUTPUT_DIR / "results.json").write_text(
    json.dumps(results, indent=4),
    encoding="utf-8",
)

## Interactive Demo

In [5]:
from emergence.generation.generator import TextGenerator
from emergence.data.tokenizer import SortingTokenizer

tokenizer = SortingTokenizer()

generator = TextGenerator.from_checkpoint(
    checkpoint_path=CHECKPOINT_DIR / "best.pt",
    config=config,
    tokenizer=tokenizer,
    device=device,
)

while True:
    prompt = input("Prompt (or 'exit'): ")

    if prompt.strip().lower() == "exit":
        break

    if not prompt.strip():
        continue

    generated_text = generator.generate(
        prompt=prompt,
        max_new_tokens=11,
        temperature=0.8,
    )

    print("\n" + generated_text)
    print("\n" + "-" * 60 + "\n")


1 2 3 4 5 | 1 2 3 4 5

------------------------------------------------------------


89 88 87 86 85 84 83 82 | 82 83 84 85 86 87 88 89

------------------------------------------------------------

